# Atlas Dinamico De Redes

Pergunta fisica: da para reconhecer o tipo de rede olhando como uma excitacao se move e chega ao alvo?

Este notebook carrega ou roda a campanha `dynamic_network_atlas`. Ele compara redes por chegada ao alvo, tempo de chegada, coerencia, entropia, pureza, espalhamento, participacao/IPR, diferenca quantico menos classico e regime fisico.

## 1. Dicionario simples

- `W/J`: irregularidade local comparada a forca de acoplamento entre nos.
- `gamma_phi/J`: taxa de embaralhamento de fase comparada a forca de acoplamento.
- `sink`: canal de chegada ao alvo.
- `arrival`: quanto da excitacao chegou ao alvo.
- `entropy`: mistura/espalhamento do estado; nao significa eficiencia sozinha.
- `quantum_minus_classical`: chegada quantica menos chegada classica no mesmo grafo.

In [ ]:
from pathlib import Path
import json
import csv
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'outputs' / 'transport_networks' / 'dynamic_network_atlas' / 'latest'
FIG = OUT / 'figures'
OUT

## 2. Parametros editaveis

Use `smoke` para testar rapido. Use `strong` para a campanha forte. Use `resume` para continuar uma campanha forte interrompida.

In [ ]:
PROFILE = 'smoke'  # troque para 'strong' ou 'resume'
RUN_NOW = False    # mude para True se quiser rodar daqui

In [ ]:
if RUN_NOW:
    import subprocess, sys
    cmd = [sys.executable, 'scripts/run_transport_dynamic_network_atlas.py', '--profile', PROFILE]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

## 3. Carregar dados

Se esta celula falhar, rode primeiro:

`python scripts/run_transport_dynamic_network_atlas.py --profile smoke`

In [ ]:
records_path = OUT / 'atlas_records.csv'
metrics_path = OUT / 'atlas_metrics.json'
assert records_path.exists(), f'Arquivo nao encontrado: {records_path}'
with records_path.open('r', encoding='utf-8', newline='') as handle:
    records = list(csv.DictReader(handle))
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
display(metrics)
records[:5]

## 4. Figuras principais

Abra cada figura e leia como um mapa fisico: familia da rede, irregularidade local, chegada ao alvo, mistura, coerencia e diferenca contra o controle classico.

In [ ]:
for name in [
    'atlas_dashboard.png',
    'arrival_by_family_heatmap.png',
    'entropy_coherence_panel.png',
    'quantum_minus_classical_map.png',
    'regime_phase_map.png',
    'signature_embedding.png',
    'family_fingerprint_radar.png',
]:
    path = FIG / name
    if path.exists():
        display(Markdown(f'### {name}'))
        display(Image(filename=str(path)))
    else:
        print('Missing:', path)

## 5. Como ler as metricas

- `best_arrival`: melhor chegada ao alvo dentro da varredura de embaralhamento de fase.
- `dephasing_gain`: quanto a melhor chegada melhorou em relacao ao caso sem embaralhamento de fase.
- `best_final_entropy`: quanto o estado ficou misturado no melhor ponto.
- `best_final_purity`: perto de 1 significa mais puro; menor significa mais misturado.
- `best_participation_ratio`: quantos nos participam de forma efetiva.
- `best_ipr`: alto significa mais localizacao; baixo significa mais espalhamento.
- `quantum_minus_classical`: positivo indica que o modelo quantico aberto chegou mais ao alvo que o controle classico.

In [ ]:
with (OUT / 'atlas_summary_by_family.csv').open('r', encoding='utf-8', newline='') as handle:
    summary = list(csv.DictReader(handle))
cols = [
    'family', 'record_count', 'best_arrival_mean', 'dephasing_gain_mean',
    'best_final_entropy_mean', 'best_final_purity_mean',
    'best_participation_ratio_mean', 'quantum_minus_classical_mean',
    'quantum_classical_verdict'
]
sorted([{key: row.get(key, '') for key in cols} for row in summary], key=lambda row: float(row['best_arrival_mean']), reverse=True)

## 6. Comparacao quantico vs classico

Regra conservadora: se a diferenca entre quantico e classico fica dentro de `0.05`, nao chamamos de assinatura quantica forte.

In [ ]:
with (OUT / 'quantum_classical_delta.csv').open('r', encoding='utf-8', newline='') as handle:
    qc = list(csv.DictReader(handle))
groups = {}
for row in qc:
    key = (row['family'], row['qc_case_label'])
    groups.setdefault(key, []).append(float(row['quantum_minus_classical']))
[{'family': key[0], 'label': key[1], 'count': len(values), 'mean_delta': sum(values)/len(values)} for key, values in sorted(groups.items())]

## 7. Regimes fisicos

Os regimes sao rotulos por limiares: coerente, assistido por ruído, localizado por irregularidade, dominado por perda, fortemente amortecido ou misto. Eles ajudam a nao depender so de impressao visual.

In [ ]:
with (OUT / 'atlas_regime_fractions.csv').open('r', encoding='utf-8', newline='') as handle:
    regimes = list(csv.DictReader(handle))
sorted(regimes, key=lambda row: (row['family'], float(row['disorder_strength_over_coupling'])))[:20]

## 8. Conclusoes permitidas

Pode afirmar: o atlas mede assinaturas dinamicas comparaveis entre redes, com controle classico e incerteza agregada.

Nao afirmar ainda: simulacao microscopica de materiais reais, prova definitiva de vantagem quantica ou que espalhamento sozinho e transporte util.

Proxima campanha sugerida: refinar as familias onde `quantum_minus_classical` fica positivo com CI95 favoravel e onde o ganho por embaralhamento de fase passa de `0.05`.